Not the most exciting chapter, but here we will import a dataset of text alongside with labels of spam/not spam.

In [2]:
import requests
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

In [3]:
if data_file_path.exists():
    print("Skipping downloads and extraction.")
else:
    
    #download file
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()

    with open(zip_path, "wb") as out_file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                out_file.write(chunk)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extracted_path)

    # give it the CSV extension
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)

    # delete the zip
    if os.path.exists(zip_path):
        os.remove(zip_path)

Skipping downloads and extraction.


You can even view the data using pandas! But since it's a tsv and not a csv, we have to use a sep='\t'

In [7]:
import pandas as pd

df = pd.read_csv(data_file_path,  sep="\t", header=None, names=['Label', 'Text'])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


And also we would like to get just s small sample of these total texts and put them toegher as a small dataset.

In [21]:
df_spam = df[df["Label"] == "spam"]

num_samples = len(df_spam)
df_real = df[df["Label"] == "ham"].sample(num_samples)

# combining together
balanced_df = pd.concat([df_real, df_spam])
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})    
balanced_df

,Label,Text
1633,0,Hello my little party animal! I just thought I...
2025,0,U having lunch alone? I now so bored...
3376,0,:)
2541,0,"They said if its gonna snow, it will start aro..."
999,0,Then ü wait 4 me at bus stop aft ur lect lar. ...
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


And for the use of traning and testing, we will od a 7/3 split for training & testing datsets.

In [22]:
split = int(len(balanced_df) * 0.7)

# first shuffle the entire DataFrame
balanced_df = balanced_df.sample(frac=1).reset_index(drop=True)
train_df = balanced_df[:split]
test_df = balanced_df[split:]

## Padding

Now, we have to realize one of the most important, any annoying issues in training with text in post-training pipielines.

Text doesn't have uniform length, but GPUs like tensors with fixed shapes, like (3, 5), or (7, 9).

Text isn't like that, if we wanted to feed the computer 8 sentences in parallel, it would wail and compliain:

- 'hello, my name is mike" (6)
- 'I like pancakes' (3)
- 'You see, GPUs are very sophisticated animals' (8)

Ideally, this should be a be (3, 8) tensor, but text is just differently lengthed! There are more research in using other formats than padding for allowing the GPU to take these different lengthed sequences into computation, but this book currently just shows this very native one.

<img src='attachments/Screenshot 2026-03-01 at 5.38.33 PM.png' width=500>


We will create a pytorch dataset class for this.

In [37]:
def tokenize_and_pad(text, tokenizer, pad_token_id, max_len):
    encoded = tokenizer.encode(text)
    
    if len(encoded) > max_len:
        encoded = encoded[:max_len]

    if len(encoded) < max_len:
        encoded = encoded + [pad_token_id] * (max_len - len(encoded))

    return encoded

In [38]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):

    def __init__(self, pandas_df, tokenizer, max_len, pad_token_id):

        self.max_len = max_len
        self.data = pandas_df.copy()
        self.data["encoded_text"] = pandas_df["Text"].apply(lambda text: tokenize_and_pad(text, tokenizer, pad_token_id, max_len))

    def __getitem__(self, index):
    
        encoded_text = self.data.iloc[index]["encoded_text"]
        label = self.data.iloc[index]["Label"]
        return torch.tensor(encoded_text, dtype=torch.long), torch.tensor(label, dtype=torch.long)
    
    def __len__(self):
        return len(self.data)

In [39]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
eot_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]
pad_token_id = eot_id

max_length = 128
train_dataset = SpamDataset(train_df, tokenizer, max_length, pad_token_id)
test_dataset = SpamDataset(test_df, tokenizer, max_length, pad_token_id)

make the dataloaders, which is of size (8, 128) each batch.

In [40]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(42)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

for inputs, targets in train_loader:
    print(f"Input Batch Dimensions {inputs.shape}")
    print(f"Target Batch Dimensions {targets.shape}")
    break

Input Batch Dimensions torch.Size([8, 128])
Target Batch Dimensions torch.Size([8])


## Initialize Model With Pre-trained Weights

In [ ]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)